In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
CAPELLA_DATASET_PATH = "/Users/fontana/Desktop/research/datafusion-agent/data/raw/20260403_capella_ieee_datacontest_2026_v01.csv"

# Load the dataset
df = pd.read_csv(CAPELLA_DATASET_PATH)

rows, cols = df.shape
print(f"Dataset shape: {rows} rows, {cols} columns")

# Display the first few rows of the dataset
df.head()

Dataset shape: 1582 rows, 32 columns


,stac_id,collect_id,datetime,start_datetime,end_datetime,center_lon,center_lat,platform,constellation,instrument_mode,...,resolution_azimuth,resolution_ground_range,pixel_spacing_range,pixel_spacing_azimuth,image_length,image_width,looks_range,looks_azimuth,orbital_plane,collection_type
0,CAPELLA_C13_SP_SLC_HH_20251105143522_202511051...,7a6bebc3-d4d6-43fb-b8b8-b2ff5f752f9f,2025-11-05T14:35:28.814850Z,2025-11-05T14:35:26.685365Z,2025-11-05T14:35:30.944336Z,-155.286305,19.421466,capella-13,capella,spotlight,...,0.44,0.27,0.25,0.41,4981.396849,10197.199779,1,1,53,spotlight
1,CAPELLA_C13_SP_SLC_HH_20251112022441_202511120...,f1e3c4b6-a64f-466a-900b-cec64250a0eb,2025-11-12T02:24:47.676882Z,2025-11-12T02:24:45.695697Z,2025-11-12T02:24:49.658068Z,-155.286169,19.420576,capella-13,capella,spotlight,...,0.44,0.28,0.26,0.41,5042.676290,10049.452929,1,1,53,spotlight
2,CAPELLA_C13_SP_GEO_HH_20251105143522_202511051...,7a6bebc3-d4d6-43fb-b8b8-b2ff5f752f9f,2025-11-05T14:35:28.814912Z,2025-11-05T14:35:22.422744Z,2025-11-05T14:35:35.207080Z,-155.286257,19.421424,capella-13,capella,spotlight,...,0.56,0.38,0.25,0.25,4981.655578,10088.212063,1,3,53,spotlight
3,CAPELLA_C13_SP_SLC_HH_20251108101220_202511081...,c2b1fe00-881f-43c9-99ed-7bad19ee492f,2025-11-08T10:12:25.262548Z,2025-11-08T10:12:23.671454Z,2025-11-08T10:12:26.853642Z,-121.870884,37.318473,capella-13,capella,spotlight,...,0.44,0.37,0.34,0.41,5010.784270,8168.277075,1,1,53,spotlight
4,CAPELLA_C13_SP_SLC_HH_20251111122850_202511111...,9d88301a-5646-4931-b81d-1129b63b2049,2025-11-11T12:28:56.582394Z,2025-11-11T12:28:54.452785Z,2025-11-11T12:28:58.712003Z,-155.286285,19.421482,capella-13,capella,spotlight,...,0.44,0.27,0.25,0.41,4981.369918,10197.050171,1,1,53,spotlight


In [4]:
# Make a copy of the original DataFrame to work with
df_copy = df.copy()

# Convert the 'timestamp' column to datetime format
df_copy['datetime_parsed'] = pd.to_datetime(df_copy['datetime'])

print(df_copy["datetime_parsed"].head())

0   2025-11-05 14:35:28.814850+00:00
1   2025-11-12 02:24:47.676882+00:00
2   2025-11-05 14:35:28.814912+00:00
3   2025-11-08 10:12:25.262548+00:00
4   2025-11-11 12:28:56.582394+00:00
Name: datetime_parsed, dtype: datetime64[us, UTC]


## Clustering

In this step we will use DBSCAN with haversine metric. Haversine is a formula that computes the actual great-circle distance between two points on a sphere, accounting for Earth's curvature.


```python
DBSCAN(eps=eps_rad, min_samples=1, metric="haversine")
```

"find clusters where points are within 5 km of each other, measuring distance properly on the Earth's surface." The coordinates get converted to radians and the eps threshold gets converted from kilometers to radians (dividing by Earth's radius) because that's the unit haversine works in.

In practice, for this dataset, the target sites are separated by hundreds or thousands of kilometers, so even simple rounding of lat/lon would have worked. But haversine is the correct general approach — it would still work if you had two monitored sites that are, say, 8 km apart and need to be distinguished.

5 / 6371 ≈ 0.000785 radians. This is the neighborhood radius DBSCAN will use — any two points closer than this are considered neighbors.

This runs the actual clustering. The three parameters mean:

* eps=eps_rad — the neighborhood radius (our 5 km, expressed in radians)

* min_samples=1 — a cluster can be formed by a single point. I set this to 1 because I don't want any points discarded as noise at this stage; even a location with only one acquisition should get its own spatial cluster. The filtering by minimum sequence length happens later.

* metric="haversine" — tells DBSCAN to measure distances using the haversine formula instead of the default Euclidean

.fit(coords_rad) feeds the radian-converted coordinates in and runs the algorithm.

In [5]:
# Get the coordinates in radians
coord_rad = np.radians(df_copy[['center_lat', 'center_lon']].values)

# instantiate earth mean radius in kilometers
earth_radius_km = 6371.0

# Define the spatial epsilon in kilometers
spatial_eps_km = 5

# Calculate the spatial epsilon in radians
spatial_eps_rad = spatial_eps_km / earth_radius_km

In [6]:
from sklearn.cluster import DBSCAN

min_samples = 1 # Minimum number of samples in a neighborhood for a point to be considered a core point

# Instantiate DBSCAN
dbscan = DBSCAN(eps=spatial_eps_rad, min_samples=min_samples, metric='haversine')

# Fit DBSCAN to the coordinate data
dbscan.fit(coord_rad)

# Get the cluster labels
cluster_labels = dbscan.labels_

# Add cluster labels to the DataFrame
df_copy['spatial_cluster'] = cluster_labels

print(f"Number of clusters found: {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")

Number of clusters found: 42


In [7]:
# Initizzlize sequence columns 
df_copy['sequence_id'] = None
df_copy['sequence_order'] = np.nan
df_copy['sequence_length'] = np.nan
seq_counter = 0

incidence_angle_threshold = 3  # Define a threshold for incidence angle difference to consider points in the same sequence
min_sequence_length = 2  # Minimum number of points in a sequence to be considered valid

# Set group keys
group_keys = ["spatial_cluster", "orbital_plane", "orbit_state", "observation_direction"]

for _, geo_group in df_copy.groupby(group_keys):
    # Sub-cluster by incidence angle using 1D DBSCAN
    inc_angles = geo_group["incidence_angle"].values.reshape(-1, 1)

    # Instantiate DBSCAN for incidence angle clustering
    dbscan_inc = DBSCAN(eps=incidence_angle_threshold, min_samples=min_sequence_length) 

    # Fit DBSCAN to the incidence angle data
    dbscan_inc.fit(inc_angles)

    # Get the cluster labels for incidence angle
    for inc_label in set(dbscan_inc.labels_):
        if inc_label == -1:
            continue  # noise points (below min_sequence_length)

        mask = dbscan_inc.labels_ == inc_label
        seq_indices = geo_group.index[mask]

        if len(seq_indices) < min_sequence_length:
            continue

        seq_counter += 1
        seq_id = f"SEQ_{seq_counter:04d}"

        # Assign temporal order within the sequence
        sub = df_copy.loc[seq_indices].sort_values("datetime_parsed")
        for rank, idx in enumerate(sub.index, start=1):
            df_copy.at[idx, "sequence_id"] = seq_id
            df_copy.at[idx, "sequence_order"] = rank
            df_copy.at[idx, "sequence_length"] = len(seq_indices)

In [9]:
 
def summarize_sequences(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce a summary table of all identified sequences.
 
    Parameters
    ----------
    df : pd.DataFrame
        Output of `identify_sequences`.
 
    Returns
    -------
    pd.DataFrame
        One row per sequence with key statistics.
    """
    seqs = df.dropna(subset=["sequence_id"])
    if seqs.empty:
        return pd.DataFrame()
 
    dt = pd.to_datetime(seqs["datetime"])
    summary = seqs.groupby("sequence_id").agg(
        n_acquisitions=("stac_id", "count"),
        n_collect_ids=("collect_id", "nunique"),
        center_lat=("center_lat", "mean"),
        center_lon=("center_lon", "mean"),
        orbital_plane=("orbital_plane", "first"),
        orbit_state=("orbit_state", "first"),
        observation_direction=("observation_direction", "first"),
        mean_incidence_angle=("incidence_angle", "mean"),
        product_types=("product_type", lambda x: ", ".join(sorted(x.unique()))),
        platforms=("platform", lambda x: ", ".join(sorted(x.unique()))),
        first_acquisition=("datetime", "min"),
        last_acquisition=("datetime", "max"),
    ).reset_index()
 
    summary["mean_incidence_angle"] = summary["mean_incidence_angle"].round(1)
    summary["center_lat"] = summary["center_lat"].round(4)
    summary["center_lon"] = summary["center_lon"].round(4)
 
    return summary.sort_values("n_acquisitions", ascending=False).reset_index(drop=True)
 

In [11]:
# display summary of identified sequences
summary_df = summarize_sequences(df_copy)
summary_df

,sequence_id,n_acquisitions,n_collect_ids,center_lat,center_lon,orbital_plane,orbit_state,observation_direction,mean_incidence_angle,product_types,platforms,first_acquisition,last_acquisition
0,SEQ_0012,198,99,34.8330,-118.0728,53,ascending,right,37.2,"GEO, SLC",capella-13,2024-11-03T15:12:50.150870Z,2025-11-08T03:24:49.064566Z
1,SEQ_0009,190,95,19.4214,-155.2863,53,descending,right,55.9,"GEO, SLC",capella-13,2025-01-08T02:08:34.973917Z,2025-11-11T12:28:56.582521Z
2,SEQ_0006,186,93,19.4209,-155.2866,53,ascending,left,52.1,"GEO, SLC",capella-13,2025-01-08T16:04:25.047451Z,2025-11-12T02:24:47.676937Z
3,SEQ_0010,176,88,37.3188,-121.8706,53,descending,right,36.6,"GEO, SLC",capella-13,2025-01-10T21:45:30.997999Z,2025-11-11T09:09:09.486448Z
4,SEQ_0051,144,72,-23.1841,118.7708,45,descending,left,40.5,"GEO, SLC","capella-10, capella-14, capella-9",2023-04-13T04:12:13.535840Z,2024-09-17T00:11:59.814430Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,SEQ_0019,2,1,34.7998,-118.0591,97,descending,right,39.1,"GEO, SLC",capella-3,2021-03-13T17:32:11.244451Z,2021-03-13T17:32:11.244606Z
77,SEQ_0018,2,1,34.8035,-118.0638,97,descending,left,33.3,"GEO, SLC",capella-3,2021-08-30T18:01:35.170710Z,2021-08-30T18:01:35.171039Z
78,SEQ_0017,2,1,34.8031,-118.0639,97,ascending,right,30.4,"GEO, SLC",capella-3,2021-09-20T05:18:16.809206Z,2021-09-20T05:18:16.809285Z
79,SEQ_0016,2,1,34.8033,-118.0629,97,ascending,left,38.0,"GEO, SLC",capella-3,2021-09-30T04:49:11.597646Z,2021-09-30T04:49:11.597742Z
